# Fine-tuning Qwen3.5-0.8B & LFM2.5-VL-1.6B — hypotheses & ablations (side-by-side)

The two Part-2 bases chosen from the full GPU comparison:
* **qwen3_5-0.8b** — strongest *genuinely sub-1B* (text + multilingual/CJK + chart),
* **lfm2_5-vl-1.6b** — best overall **and** fastest (~1 s/sample).

This notebook (1) visualises the **baseline gaps side-by-side**, (2) states the **hypothesis** to
close each gap (which module to adapt → which ablation arm), and (3) runs an **ablation study per
section** with a **before/after side-by-side** bar for both models. Training/eval cells need a GPU;
run them with `scripts/run_ablation.py`, which writes `docs/results/ablation_results.json` that the
plots below read. Run order: Runtime → Run all.

In [ ]:
# --- install docvlm_eval (fresh env e.g. Colab: clone+checkout; then editable install) ---
import os, sys, subprocess, importlib
from pathlib import Path

def _repo_root():
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "pyproject.toml").exists():
            return c
    return None

root = _repo_root()
if root is None:                       # fresh environment (Colab/Kaggle): clone the repo
    subprocess.run(["git", "clone", "https://github.com/SangbumChoi/OCR.git"], check=False)
    root = Path("OCR")
    subprocess.run(["git", "-C", str(root), "checkout", "claude/new-session-w79q0i"], check=False)
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
os.chdir(root)                         # cwd is now the repo root
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[newvlms,finetune,synth]"], check=True)

# The editable-install .pth is only read at interpreter startup, so a running kernel can't import
# the package until we add src/ to sys.path ourselves (avoids "No module named docvlm_eval").
src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
import docvlm_eval
print("docvlm_eval ready from", docvlm_eval.__file__)


## 1. Baseline gaps — side-by-side (from the full GPU run)

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT/"scripts").exists() and (ROOT.parent/"scripts").exists():
    ROOT = ROOT.parent
MODELS = ["qwen3_5-0.8b", "lfm2_5-vl-1.6b"]
COLOR = {"qwen3_5-0.8b": "#d7791d", "lfm2_5-vl-1.6b": "#2c7bb6"}

# measured baselines from notebooks/colab_full_comparison.ipynb (directional: 1 sample/axis for the
# capability cells; spatial = control-robust PASS; custom-eval/text = multi-sample).
BASE = {
  "qwen3_5-0.8b":   {"T1":1.0,"T2":1.0,"H1":1.0,"H2":0.0,"H3":1.0,"L1":0.0,
                     "spatial_pass":3, "text":0.777, "spot_iou":0.008, "rot180":0.714, "lat_s":13.9},
  "lfm2_5-vl-1.6b": {"T1":1.0,"T2":1.0,"H1":1.0,"H2":1.0,"H3":1.0,"L1":0.0,
                     "spatial_pass":5, "text":0.829, "spot_iou":0.229, "rot180":1.0, "lat_s":0.98},
}
AXES = ["T1","T2","H1","H2","H3","L1"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(len(AXES)); w = 0.38
for i, m in enumerate(MODELS):
    ax[0].bar(x + (i-0.5)*w, [BASE[m][a] for a in AXES], w, label=m, color=COLOR[m])
ax[0].set_xticks(x); ax[0].set_xticklabels(AXES); ax[0].set_ylim(0,1.05)
ax[0].set_title("Capability probe (T·text / H·reasoning / L·location)"); ax[0].legend(fontsize=8)
ax[0].axvspan(2.5, 3.5, color="red", alpha=0.07); ax[0].axvspan(4.5, 5.5, color="red", alpha=0.07)

extra = ["spatial_pass/7","text","spot_iou","rot180"]
vals = lambda m: [BASE[m]["spatial_pass"]/7, BASE[m]["text"], BASE[m]["spot_iou"], BASE[m]["rot180"]]
x2 = np.arange(len(extra))
for i, m in enumerate(MODELS):
    ax[1].bar(x2 + (i-0.5)*w, vals(m), w, label=m, color=COLOR[m])
ax[1].set_xticks(x2); ax[1].set_xticklabels(extra, fontsize=8); ax[1].set_ylim(0,1.05)
ax[1].set_title("Context-robust + custom-eval signals")
plt.tight_layout(); plt.show()

print("Shared lacks  -> L1 grounding ~0 ; L4 box-tracking = 0 (both) ; 180-deg rotation")
print("qwen3.5 lacks -> H2 relational-compare = 0 ; latency ~14s/sample")
print("lfm2.5  lacks -> grounding (spot-IoU 0.23, best but low) ; box-tracking")

## 2. Hypotheses — which module to adapt to close each gap

Capability is **module-localised** (see `docs/report/research_novelty.md`), so each gap is attacked
where it physically lives in the encoder→connector→LLM stack:

| Gap (who) | Root module to adapt | Ablation arm | Why | Expected effect |
| --- | --- | --- | --- | --- |
| **L1 grounding / spotting** (both) | vision + **connector** | A1 + A5(connector) | "where" is geometric; the connector serialises positions | spot-IoU ↑, `cap_ground` ↑ |
| **L4 box-tracking** (both) | vision + connector | A1 (sequential boxes) | tracking = repeated localisation | L4 PASS emerges |
| **H2 relational reasoning** (qwen3.5) | **LLM** attn + mlp | A2 + A5(llm) | multi-region compare is an LM computation | `cap_integ_rel` ↑ |
| **180° rotation** (both) | vision + input | A7 (orientation aug) | legibility/orientation is an encoder property | rot-180 retention ↑ |
| **latency** (qwen3.5) | — (decode) | A6 (shorter targets / r) | fewer tokens, smaller rank | latency ↓ |

The supervision for A1/A2/A4/A7 is exactly the model-free GT the generator now emits
(`ask_where/region/count/aggregate` + rationales; `configs/synth_data.yaml`).

## 3. Ablation study — per section (GPU; side-by-side before/after)

In [ ]:
RESULTS = ROOT / "docs" / "results" / "ablation_results.json"

def _load():
    return json.loads(RESULTS.read_text()) if RESULTS.exists() else {"models": {}}

def cmd(arm, placement="all"):
    """Print the GPU command that fills this arm for both models."""
    print(f"!python scripts/run_ablation.py --models {' '.join(MODELS)} "
          f"--arm {arm} --placement {placement}")

def side_by_side(arm_key, title, axis=None):
    """Grouped bars: baseline vs <arm> overall score (or one by_answer_type axis), per model.
    Reads ablation_results.json; shows a 'pending' note for arms not run yet."""
    d = _load().get("models", {})
    fig, ax = plt.subplots(figsize=(7, 4)); w = 0.38; x = np.arange(2)
    any_data = False
    for i, m in enumerate(MODELS):
        runs = d.get(m, {})
        def score(key):
            s = runs.get(key)
            if not s: return None
            return s["by_answer_type"].get(axis, {}).get("score") if axis else s.get("score")
        b, a = score("baseline"), score(arm_key)
        if b is not None or a is not None: any_data = True
        ax.bar(x + (i-0.5)*w, [b or 0, a or 0], w, label=m, color=COLOR[m])
    ax.set_xticks(x); ax.set_xticklabels(["baseline", arm_key]); ax.set_ylim(0, 1.05)
    ax.set_title(title + ("" if any_data else "  (run the cell above on GPU to populate)"))
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### A1 — spotting supervision

Add `value + [x1,y1,x2,y2]` targets (ask_where/region). Hypothesis: grounding lives in vision+connector → adapting the **connector** lifts `cap_ground`/spot-IoU on both models.

In [ ]:
cmd("A1_spotting_on", "connector")
side_by_side("A1_spotting_on:connector", "A1 — spotting supervision", axis='L1')

### A2 — reasoning supervision

Add `rationale → answer` targets. Hypothesis: relational/numeric reasoning is an **LLM** computation → adapting `llm_attn` (+mlp) closes qwen3.5's H2 gap; lfm2.5 already strong.

In [ ]:
cmd("A2_reasoning_on", "llm_attn")
side_by_side("A2_reasoning_on:llm_attn", "A2 — reasoning supervision", axis='H2')

### A4 — multilingual mix

Train ko+en (then other pairs). Hypothesis: new vocabulary/script is stored in the **LLM MLP/embeddings**; related scripts transfer, distant interfere at fixed capacity.

In [ ]:
cmd("A4_ko_en", "llm_mlp")
side_by_side("A4_ko_en:llm_mlp", "A4 — multilingual mix", axis=None)

### A5 — LoRA placement

Same A1 data, sweep the placement group {vision|connector|llm_attn|llm_mlp|all}. Hypothesis: grounding gains concentrate in vision+connector, reasoning in the LLM — a capability×module interaction (resolved by introspection, `finetune.lora_vlm.resolve_lora_targets`).

In [ ]:
cmd("A1_spotting_on", "vision")
side_by_side("A1_spotting_on:vision", "A5 — LoRA placement", axis='L1')

### A7 — preprocessing / orientation

Higher-res dynamic tiling + orientation augmentation. Hypothesis: small-text recognition is an encoder+resolution property → `cap_text`/small-text ↑ and 180° rotation retention ↑.

In [ ]:
cmd("A7_dynamic_tiling", "all")
side_by_side("A7_dynamic_tiling:all", "A7 — preprocessing / orientation", axis='T1')

## 4. Cumulative staircase — both models

In [ ]:
order = ["baseline", "A7_dynamic_tiling:all", "A1_spotting_on:connector",
         "A2_reasoning_on:llm_attn", "A4_ko_en:llm_mlp"]
d = _load().get("models", {})
fig, ax = plt.subplots(figsize=(10, 4.5))
for m in MODELS:
    runs = d.get(m, {})
    ys = [runs.get(k, {}).get("score") for k in order]
    xs = [i for i, y in enumerate(ys) if y is not None]
    ax.plot(xs, [ys[i] for i in xs], "-o", label=m, color=COLOR[m]) if xs else None
ax.set_xticks(range(len(order))); ax.set_xticklabels([o.split(":")[0] for o in order], rotation=20, fontsize=8)
ax.set_ylabel("probe score"); ax.set_title("Cumulative ablation staircase (fill via run_ablation.py)")
ax.legend(); plt.tight_layout(); plt.show()
print("Each step stacks the winning arm; a flat/negative step is itself a finding (drop it).")